# Локальная диаризация аудио с NVIDIA NeMo Sortformer


### Установка зависимостей
После первой установки выбрать Среда выполнения -> Перезапустить сеанс, затем продолжить с ячейки инициализации ниже, не запуская установку повторно.


In [1]:
%pip install -q Cython packaging "nemo_toolkit[asr]==2.7.0" pandas memory_profiler


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Инициализация девайса и библиотек
После перезапуска начать отсюда.

In [1]:
import json
import re
import subprocess
from pathlib import Path
from time import perf_counter

import pandas as pd
from memory_profiler import memory_usage
import torch
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден. Будет использован CPU.")


GPU не найден. Будет использован CPU.


### Выбор локального аудиофайла

Указать путь к одному локальному аудиофайлу или медиаконтейнеру с аудиодорожкой в переменной AUDIO_PATH.


In [2]:
AUDIO_PATH = Path(r"content/audio_files/ES2002a_300sec.wav")
audio_path = AUDIO_PATH.expanduser().resolve()
audio_name = audio_path.name
allowed_extensions = {
    ".3g2", ".3gp", ".aac", ".ac3", ".aif", ".aifc", ".aiff",
    ".amr", ".ape", ".au", ".avi", ".awb", ".caf", ".dts",
    ".eac3", ".flac", ".flv", ".gsm", ".m2ts", ".m4a", ".m4b",
    ".mka", ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpc",
    ".mpeg", ".mpg", ".mts", ".oga", ".ogg", ".opus", ".ra",
    ".rm", ".snd", ".spx", ".tak", ".ts", ".tta", ".wav",
    ".wave", ".webm", ".wma", ".wv",
}
if audio_path.suffix.lower() not in allowed_extensions:
    raise ValueError(
        f"Неподдерживаемый формат {audio_path.suffix or 'без расширения'}. "
        "Выберите аудиофайл или медиаконтейнер с поддерживаемой аудиодорожкой."
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise ValueError(f"Локальный файл отсутствует или пуст: {audio_path}")
print(f"Выбран файл: {audio_path} ({audio_path.stat().st_size / 1024 / 1024:.2f} МБ)")


Выбран файл: D:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\content\audio_files\ES2002a_300sec.wav (9.16 МБ)


### Подготовка аудио


In [3]:
prepared_dir = Path("content/prepared_audio")
prepared_dir.mkdir(parents=True, exist_ok=True)
prepared_path = prepared_dir / f"{audio_path.stem}_mono_16khz.wav"
command = [
    "ffmpeg", "-v", "error", "-y", "-i", str(audio_path),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(prepared_path),
]
try:
    conversion = subprocess.run(command, capture_output=True, text=True, check=False)
except FileNotFoundError as exc:
    raise RuntimeError("FFmpeg not found.") from exc

if conversion.returncode != 0 or not prepared_path.is_file() or prepared_path.stat().st_size <= 44:
    details = (conversion.stderr or "FFmpeg error").strip()[-1000:]
    raise RuntimeError(
        "Не удалось прочитать или преобразовать аудио."
        f"Сообщение FFmpeg: {details}"
    )
print(f"Аудио подготовлено: {prepared_path} (mono, 16 кГц, WAV)")


Аудио подготовлено: content\prepared_audio\ES2002a_300sec_mono_16khz.wav (mono, 16 кГц, WAV)


### Загрузка модели


In [4]:
from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_ID = "nvidia/diar_sortformer_4spk-v1"
try:
    diar_model = SortformerEncLabelModel.from_pretrained(MODEL_ID)
    diar_model.to(DEVICE)
    diar_model.eval()
except Exception as exc:
    message = str(exc)
    raise RuntimeError(f"Не удалось загрузить модель: {message}") from exc
print(f"Модель загружена на {DEVICE}.")


d:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-08-13 18:18:55 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-08-13 18:18:59 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
 

[NeMo I 2026-08-13 18:19:02 save_restore_connector:285] Model SortformerEncLabelModel was successfully restored from C:\Users\Vovaf\.cache\huggingface\hub\models--nvidia--diar_sortformer_4spk-v1\snapshots\9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08\diar_sortformer_4spk-v1.nemo.
Модель загружена на cpu.


### Выполнение диаризации

Модель автоматически определяет до четырёх говорящих. Число говорящих вручную не задаётся.


In [5]:
def run_diarization():
    with torch.inference_mode():
        return diar_model.diarize(
            audio=str(prepared_path),
            batch_size=1,
        )

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
diarization_started_at = perf_counter()

try:
    if DEVICE.type == "cuda":
        predicted_segments = run_diarization()
    else:
        memory_values, predicted_segments = memory_usage(
            run_diarization,
            retval=True,
            interval=0.1,
        )
except (torch.cuda.OutOfMemoryError, MemoryError) as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "Во время диаризации закончилась оперативная или видеопамять. "
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Диаризация завершилась ошибкой: {exc}."
    ) from exc

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    used_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
    memory_measurement_name = "Пиковое использование VRAM"
else:
    used_memory_gib = max(memory_values) / 1024
    memory_measurement_name = "Пиковое использование RAM"
diarization_elapsed_seconds = perf_counter() - diarization_started_at

raw_segments = predicted_segments
if (
    isinstance(raw_segments, list)
    and len(raw_segments) == 1
    and isinstance(raw_segments[0], list)
):
    raw_segments = raw_segments[0]

def normalize_speaker(value):
    match = re.search(r"(\d+)$", str(value))
    if not match:
        raise ValueError(f"Не удалось определить номер говорящего из {value!r}.")
    return f"SPEAKER_{int(match.group(1)):02d}"

segments = []
for item in raw_segments:
    if isinstance(item, str):
        parts = item.strip().replace(",", " " ).split()
        if len(parts) < 3:
            raise ValueError(f"Неожиданный сегмент Sortformer: {item!r}")
        start, end, speaker = parts[0], parts[1], parts[2]
    elif isinstance(item, (list, tuple)) and len(item) >= 3:
        start, end, speaker = item[0], item[1], item[2]
    else:
        raise ValueError(f"Неожиданный сегмент Sortformer: {item!r}")
    start = float(start)
    end = float(end)
    if end > start:
        segments.append(
            {
                "start": round(start, 3),
                "end": round(end, 3),
                "speaker": normalize_speaker(speaker),
            }
        )
segments.sort(key=lambda item: (item["start"], item["end"], item["speaker"]))
if not segments:
    raise RuntimeError(
        "В записи не обнаружена речь."
    )
print(f"Диаризация завершена. Найдено сегментов: {len(segments)}")


[NeMo I 2026-08-13 18:19:10 vad_utils:81] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-08-13 18:19:10 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: num_spks,session_len_sec,soft_label_thres
Diarizing: 1it [00:50, 50.61s/it]

Диаризация завершена. Найдено сегментов: 109


### Просмотр и сохранение временной разметки

Сегменты выводятся в таблице в минутах, а временная разметка сохраняется локально в JSON (в секундах).


In [ ]:
def format_minutes(seconds):
    minutes, remaining_seconds = divmod(float(seconds), 60)
    return f"{int(minutes):02d}:{remaining_seconds:06.3f}"

table = pd.DataFrame(
    [
        (format_minutes(item["start"]), format_minutes(item["end"]), item["speaker"])
        for item in segments
    ],
    columns=["Начало", "Окончание", "Говорящий"],
)
print(
    f"Время диаризации: {format_minutes(diarization_elapsed_seconds)} "
    f"({diarization_elapsed_seconds:.3f} с)"
)
print(f"{memory_measurement_name}: {used_memory_gib:.3f} GiB")
with pd.option_context("display.max_rows", None):
    display(table)

result = {"audio_file": audio_name, "segments": segments}
results_dir = Path("content/diarization_json_files/local/nemo/diar_sortformer_4spk-v1")
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / Path(audio_name).with_suffix(".json").name
with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Результат сохранён: {result_path.resolve()}")


Время диаризации: 00:51.837 (51.837 с)
Пиковое использование RAM: 3.481 GiB


,Начало,Окончание,Говорящий
0,00:00.480,00:01.040,SPEAKER_00
1,00:03.600,00:04.000,SPEAKER_00
2,00:06.400,00:09.520,SPEAKER_00
3,00:09.920,00:10.560,SPEAKER_00
4,00:12.480,00:12.880,SPEAKER_00
5,00:14.240,00:16.960,SPEAKER_00
6,00:18.560,00:20.560,SPEAKER_00
7,00:21.040,00:25.120,SPEAKER_00
8,00:24.960,00:25.280,SPEAKER_01
9,00:26.000,00:27.520,SPEAKER_00


Результат сохранён: D:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\content\diarization_json_files\nemo\diar_sortformer_4spk-v1\ES2002a_300sec.json
